# Axisymmetric high-order heat: a reproducible field check
Radia uses standard NGSolve H1 for an axis-touching thermal cylinder without imposing zero axis temperature. This example reproduces a quadratic transient, checks conservative P1-source/Q2-temperature transfer and displays separately recorded MEX/Simulink results. The **Radia IH MCP tool family** owns the live workflow; this notebook is not a production GUI or a CAD/VOL acceptance.

Surface-impedance electromagnetic modeling is background to the separate loss calculation (Yuferev and Ida, `YuferevIda2009`, canonical bibliography), not validation of this thermal test. The [ESIM showcase](induction_heating_demo_showcase.ipynb) covers that distinct method.

## Equation and exact solution
For constant conductivity $k$, density $\rho$ and heat capacity $c_p$,
$$\rho c_p\partial_tT-\frac1r\partial_r(rk\partial_rT)-\partial_z(k\partial_zT)=0.$$
Multiplication by $2\pi r v$ gives $M\dot t+Kt=f$, with $M_{ij}=\int 2\pi r\rho c_p\phi_i\phi_j$ and $K_{ij}=\int 2\pi r k\nabla\phi_i\cdot\nabla\phi_j$. Positive inward boundary flux enters $f_i=\int_{\Gamma_q}2\pi r q_{in}\phi_i$.

Choose $T(r,t)=T_0+a r^2+4ka t/(\rho c_p)$ in a cylinder of radius and height 25 mm. Insulated end caps and outer-wall flux $q_{in}=2kaR$ produce this solution. Its axis derivative is zero, but its axis temperature is not. Backward Euler is exact in time for this linear-in-time solution; Q2 represents its spatial polynomial. This does not prove convergence for arbitrary fields.

The mesh is created in memory, never passed through `NgMesh.Save()` or post-load curving. The production assembly kernel is loaded explicitly from this checkout without changing an editable installation. The notebook-local helper below assembles a second independent NGSolve step and checks the production CSR/mixed-load matrices against it.

In [1]:
from pathlib import Path
import sys, json
from ngsolve.webgui import Draw
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'src/radia/simulink/ih_operator_assembly.py').is_file())
sys.path.insert(0, str(root))
sys.path.insert(0, str(root/'docs/induction_heating'))
from axisymmetric_thermal_example import reproduce
mesh, temperature, metrics = reproduce(root)
print(json.dumps(metrics, indent=2))
Draw(mesh, name='axisymmetric_meridian_mesh', draw_surf=True, draw_vol=False, width=650, height=450)


{
  "NGSolve": "6.2.2606",
  "assembly_sha256": "d0d5fc1a640672395865b5556f56595d854e71cfffeef59fa4d24e1ddb4c60eb",
  "example_sha256": "4c3b35efb3e6214615cc0aca91d7a81215c607ce721b176e50b1395027fd796d",
  "temperature_DOFs": 361,
  "steps": 100,
  "heat_power_W": 91.49888603580273,
  "power_relative_error": 2.220446049250313e-16,
  "capacity_relative_error": 5.88418203051333e-15,
  "max_exact_relative_rise_L2": 1.1151730962095437e-13,
  "matrix_vs_independent_RMS_K": 1.8842097939727618e-12,
  "scope": "LAB in-memory algebra reproduction, not production CAD/VOL acceptance"
}


WebGLScene

## Temperature and coefficient semantics
Temperature is Q2 on this quadrilateral mesh; the heat source is P1. The mixed weak form conserves power through $c^Tf=P$, where $c$ represents the constant function. Initialization is $T_0c$, and ambient convection uses $Cc$, not row sums assuming every coefficient equals one. Native temperature coefficients are not nodal Kelvin values. Monitor extrema are mapped-quadrature samples, not certified global extrema.

The following field is the independently assembled NGSolve temperature at 10 seconds. Both mesh and field views retain physical coordinates.

In [2]:
Draw(temperature, mesh, name='temperature_K_at_10s', draw_surf=True, draw_vol=False, autoscale=True, width=650, height=450)


WebGLScene

## Recorded native and tracked-model checks
These JSON records are loaded, **not rerun as MATLAB calculations here**. Source/native hashes and scope remain in the records. Nanokelvin agreement measures implementation consistency, not physical IH accuracy; the independent workpiece electromagnetic comparison has its separate application-specific 2% bound.

The native record is the quadratic boundary-heated case above. The tracked-model record is a separate uniform-boundary-heating case with 361 temperature DOFs and 101 samples.

In [3]:
evidence = root/'validation_test/induction_heating'
native = json.loads((evidence/'ih_axisym_p2_exact_native_20260915.json').read_text(encoding='utf-8'))
tracked = json.loads((evidence/'ih_tracked_p2_20260915.json').read_text(encoding='utf-8'))
assert native['passed'] and tracked['passed']
assert native['handles_before'] == native['handles_after'] == 0
assert native['cases'][0]['max_temperature_difference_K'] < 1e-7
assert tracked['max_temperature_error_K'] < 1e-7
print(json.dumps({'native_max_difference_K': native['cases'][0]['max_temperature_difference_K'], 'native_source_commit': native['native_build']['source_commit'], 'tracked_max_difference_K': tracked['max_temperature_error_K'], 'tracked_samples': tracked['sample_count']}, indent=2))


{
  "native_max_difference_K": 9.26538364019496e-09,
  "native_source_commit": "71c5a2869b9605e1e24789de0be016db226fc21e",
  "tracked_max_difference_K": 9.727482883116307e-09,
  "tracked_samples": 101
}


## Limits and remaining acceptance
This verifies an axisymmetric H1 field, conservative source handoff and a manufactured transient. It does not certify unmocked production CAD/VOL-to-EM-to-heat coupling, arbitrary boundary selections, temperature-dependent nonlinear BH, high-order periodic rotation or the received-case sixfold power ratio. Main integration ([PR #280](https://github.com/ksugahar/Radia/pull/280)) is not four-host release acceptance. See [architecture and acceptance](../IH_THERMAL_WORKFLOW.md) and the IH MCP manual.

### Background reference
S. V. Yuferev and N. Ida, *Surface Impedance Boundary Conditions: A Comprehensive Approach*, CRC Press, 2009 (`YuferevIda2009` in the canonical parent bibliography). The thermal solution and matrix identities above are derived explicitly here.